In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:

shard_inx = [0,1,2]
simple_path = [f'/home/cavin/jt/benchmark/data/crc/VisiumHD_P2_shard_{shard_inx}.h5ad' for shard_inx in shard_inx]
cell_emb_col = 'X_pca'
pca_components = 50
random_seed=2026


In [3]:
adata = [sc.read_h5ad(path) for path in simple_path]
adata

[AnnData object with n_obs × n_vars = 166823 × 2000
     obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
     var: 'ensembl_id', 'gene_name'
     uns: 'spatial'
     obsm: 'spatial',
 AnnData object with n_obs × n_vars = 166823 × 2000
     obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
     var: 'ensembl_id', 'gene_name'
     uns: 'spatial'
     obsm: 'spatial',
 AnnData object with n_obs × n_vars = 166824 × 2000
     obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
     var: 'ensembl_id', 'gene_name'
     uns: 'spatial'
     obsm: 'spatial']

In [4]:
var = adata[0].var
adata_concated = sc.concat(adata, axis=0)
adata_concated.var = var
adata_concated

AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    obsm: 'spatial'

In [5]:
sc.pp.pca(adata_concated, n_comps=pca_components,random_state=random_seed)
adata_concated

AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'pca'
    obsm: 'spatial', 'X_pca'
    varm: 'PCs'

In [7]:
emb = adata_concated.obsm['X_pca']
emb.shape

(500470, 50)

In [8]:
emb_df = pd.DataFrame(
    emb, 
    index=adata_concated.obs_names,
    columns=[f"PCA_dim_{i}" for i in range(emb.shape[1])]
)
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_no_annotations/PCA_human_CRC_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")